# Next Bill Prediction — Simple Model Training
This notebook trains the model used by the Streamlit dashboard. It saves only one final model file.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## 1. Load and check the data

In [ ]:
df = pd.read_csv("../data/billings.csv")
df["billing_month"] = pd.to_datetime(df["billing_month"])
df["signup_date"] = pd.to_datetime(df["signup_date"])
df = df.sort_values(["customer_id", "billing_month"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Customers:", df["customer_id"].nunique())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicates:", int(df.duplicated().sum()))
display(df.head())

## 2. Concise EDA

In [ ]:
display(df.describe(numeric_only=True).T.round(2))

df.groupby("billing_month")["bill_amount"].mean().plot(marker="o", figsize=(10,4), title="Average Monthly Bill")
plt.ylabel("Average Bill")
plt.show()

display(df.groupby("plan_type")["bill_amount"].agg(["count", "mean", "median"]).round(2))

## 3. Historical feature engineering
We predict the next month using information available in the current and earlier months.

In [ ]:
g = df.groupby("customer_id", group_keys=False)

for k in [1, 2, 3, 6]:
    df[f"bill_lag_{k}"] = g["bill_amount"].shift(k)
    df[f"usage_lag_{k}"] = g["usage_units"].shift(k)

for w in [3, 6]:
    df[f"bill_roll_mean_{w}"] = g["bill_amount"].transform(lambda s: s.rolling(w, min_periods=2).mean())
    df[f"bill_roll_std_{w}"] = g["bill_amount"].transform(lambda s: s.rolling(w, min_periods=2).std())
    df[f"usage_roll_mean_{w}"] = g["usage_units"].transform(lambda s: s.rolling(w, min_periods=2).mean())
    df[f"usage_roll_std_{w}"] = g["usage_units"].transform(lambda s: s.rolling(w, min_periods=2).std())

df["bill_vs_roll6"] = df["bill_amount"] / df["bill_roll_mean_6"]
df["usage_vs_roll6"] = df["usage_units"] / df["usage_roll_mean_6"]
df["month"] = df["billing_month"].dt.month
df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)
df["tenure_months"] = ((df["billing_month"].dt.year - df["signup_date"].dt.year) * 12 +
                        df["billing_month"].dt.month - df["signup_date"].dt.month).clip(lower=0)

df["next_bill_amount"] = g["bill_amount"].shift(-1)
df["target_month"] = g["billing_month"].shift(-1)

## 4. Features and chronological split

In [ ]:
num_features = [
    "bill_amount", "usage_units", "peak_usage_share", "unit_price", "fixed_charge",
    "tariff_change_flag", "support_tickets", "household_size", "temperature_avg",
    "month", "sin_month", "cos_month", "tenure_months",
    "bill_lag_1", "bill_lag_2", "bill_lag_3", "bill_lag_6",
    "usage_lag_1", "usage_lag_2", "usage_lag_3", "usage_lag_6",
    "bill_roll_mean_3", "bill_roll_mean_6", "bill_roll_std_3", "bill_roll_std_6",
    "usage_roll_mean_3", "usage_roll_mean_6", "usage_roll_std_3", "usage_roll_std_6",
    "bill_vs_roll6", "usage_vs_roll6"
]
cat_features = ["plan_type", "customer_segment", "region"]
features = num_features + cat_features

model_df = df.dropna(subset=["next_bill_amount", "target_month"]).copy()
train = model_df[model_df["target_month"] < pd.Timestamp("2025-11-01")].copy()
test = model_df[model_df["target_month"] >= pd.Timestamp("2025-11-01")].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Test months:", sorted(test["target_month"].dt.strftime("%Y-%m").unique()))

## 5. Train all models together

In [ ]:
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=120, min_samples_leaf=2, max_features=0.8, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, reg_lambda=2, n_jobs=-1, random_state=42, objective="reg:squarederror"),
    "LightGBM": LGBMRegressor(n_estimators=150, num_leaves=20, learning_rate=0.05, reg_lambda=2, random_state=42, verbosity=-1)
}

def metrics(y, pred):
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "WAPE_%": np.abs(y - pred).sum() / np.abs(y).sum() * 100,
        "R2": r2_score(y, pred)
    }

rows = []
fitted = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(train[features], train["next_bill_amount"])
    pred = pipe.predict(test[features])
    rows.append({"Model": name, **metrics(test["next_bill_amount"], pred)})
    fitted[name] = pipe

scores = pd.DataFrame(rows).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
display(scores.round(4))

## 6. Select the strongest model and tune LightGBM simply

In [ ]:
train_fit = train[train["target_month"] < pd.Timestamp("2025-10-01")].copy()
valid = train[train["target_month"] >= pd.Timestamp("2025-10-01")].copy()

parameter_options = [
    {"n_estimators":150, "num_leaves":15, "learning_rate":0.03, "min_child_samples":15, "reg_lambda":1},
    {"n_estimators":200, "num_leaves":15, "learning_rate":0.03, "min_child_samples":20, "reg_lambda":2},
    {"n_estimators":250, "num_leaves":20, "learning_rate":0.03, "min_child_samples":30, "reg_lambda":5}
]

best_params = None
best_validation_rmse = float("inf")
for params in parameter_options:
    candidate = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LGBMRegressor(**params, random_state=42, verbosity=-1))
    ])
    candidate.fit(train_fit[features], train_fit["next_bill_amount"])
    pred = candidate.predict(valid[features])
    rmse = np.sqrt(mean_squared_error(valid["next_bill_amount"], pred))
    if rmse < best_validation_rmse:
        best_validation_rmse = rmse
        best_params = params

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(**best_params, random_state=42, verbosity=-1))
])
final_model.fit(train[features], train["next_bill_amount"])
final_pred = final_model.predict(test[features])
print("Best parameters:", best_params)
print("Final test metrics:", metrics(test["next_bill_amount"], final_pred))

## 7. Save only the final model

In [ ]:
bundle = {
    "model": final_model,
    "best_model_name": "LightGBM",
    "features": features,
    "num_features": num_features,
    "cat_features": cat_features,
    "business_high_bill_threshold": 2.5,
    "best_params": best_params,
    "categorical_options": {
        "plan_type": sorted(df["plan_type"].dropna().unique().tolist()),
        "customer_segment": sorted(df["customer_segment"].dropna().unique().tolist()),
        "region": sorted(df["region"].dropna().unique().tolist())
    }
}
joblib.dump(bundle, "../models/best_model.joblib")
print("Saved: ../models/best_model.joblib")